# Install Packages

In [24]:
!pip install capellambse jinja2 weasyprint

# Load Model

In [25]:
from capellambse import MelodyModel

model_path = r"C:\Users\Boody\OneDrive\Desktop\Repos\ai4se-irs-generator\data\SDR.aird"
model = MelodyModel(model_path)

print("Model loaded successfully")

Model loaded successfully


## Imports

In [26]:
import os
os.add_dll_directory(r"C:\Program Files\GTK3-Runtime Win64\bin")

<AddedDllDirectory('C:\\Program Files\\GTK3-Runtime Win64\\bin')>

In [27]:
from jinja2 import Template
from weasyprint import HTML

# Identify External Functional Exchanges

In [28]:
print("\n--- Re-identifying External Functional Exchanges ---")

# Helper function to get all UUIDs of elements a functional element (port or function) is allocated to
def get_allocated_uuids_for_functional_element(functional_element):
    """
    Given a FunctionalPort or Function, return a set of UUIDs of the
    Components/Actors it is allocated to. This version iteratively traverses the
    'owner' relationship for SystemFunctions to find their top-level hosting component.
    """
    uuids = set()
    if not functional_element:
        return uuids

    current_func_or_port = None
    # If it's a FunctionalPort (either input or output), its owner is the Function
    if hasattr(functional_element, 'xtype') and \
       ('FunctionOutputPort' in functional_element.xtype or 'FunctionInputPort' in functional_element.xtype):
        current_func_or_port = functional_element.owner
    # If it's already a Function (e.g., SystemFunction), it's the element itself
    elif hasattr(functional_element, 'xtype') and functional_element.xtype.endswith(':Function'):
        current_func_or_port = functional_element
    else:
        return uuids

    current_element_in_traversal = current_func_or_port

    # Iterate up the owner chain until a non-function owner (or no owner) is found
    while current_element_in_traversal:
        # Check if the current element is a SystemComponent or Actor
        xtype = getattr(current_element_in_traversal, 'xtype', '')
        if 'SystemComponent' in xtype or 'Actor' in xtype:
            if hasattr(current_element_in_traversal, 'uuid'):
                uuids.add(current_element_in_traversal.uuid)
            break # Found the ultimate host, exit loop for this functional element

        # Also check direct 'allocated_to' property on the current function (less common for SA functions, but good to check)
        if hasattr(current_element_in_traversal, 'allocated_to') and current_element_in_traversal.allocated_to:
            for allocation_target in current_element_in_traversal.allocated_to:
                if hasattr(allocation_target, 'uuid'):
                    uuids.add(allocation_target.uuid)

        # Move up to the parent (owner) for the next iteration
        if hasattr(current_element_in_traversal, 'owner') and current_element_in_traversal.owner:
            current_element_in_traversal = current_element_in_traversal.owner
        else:
            break
    return uuids

# 1. Identify the UUID of the main 'System' component
main_system_component_uuid = None
if hasattr(model.sa, 'root_component') and model.sa.root_component:
    main_system_component_uuid = model.sa.root_component.uuid
    print(f"Main 'System' component UUID: {main_system_component_uuid}")
else:
    print("Could not find the main 'System' component (model.sa.root_component).")

# 2. Create a set of UUIDs for all external boundary elements
external_boundary_uuids = set()

# Add UUIDs of all system actors
if hasattr(model.sa, 'all_actors') and model.sa.all_actors:
    for actor in model.sa.all_actors:
        if actor.uuid != main_system_component_uuid: # Ensure 'System' is not mistakenly added if it's an actor
            external_boundary_uuids.add(actor.uuid)

# Add UUIDs of all system components that are NOT the main 'System' component
# Note: model.sa.root_component is usually of xtype 'org.polarsys.capella.core.data.ctx:SystemComponent'
if hasattr(model.sa, 'all_components') and model.sa.all_components:
    for component in model.sa.all_components:
        if component.uuid != main_system_component_uuid:
            external_boundary_uuids.add(component.uuid)

print(f"Number of identified external boundary elements: {len(external_boundary_uuids)}")

# 3. Get all functional exchanges
all_functional_exchanges = []
if hasattr(model.sa, 'all_function_exchanges') and model.sa.all_function_exchanges:
    all_functional_exchanges = list(model.sa.all_function_exchanges)

# 4. Initialize an empty list called external_functional_exchanges
external_functional_exchanges = []

# 5. Iterate through each functional exchange
if main_system_component_uuid:
    for fe in all_functional_exchanges:
        source_host_uuids = get_allocated_uuids_for_functional_element(fe.source)
        target_host_uuids = get_allocated_uuids_for_functional_element(fe.target)

        is_external_to_system = False

        # Check if the functional exchange crosses the system boundary
        # Condition 1: Source is external, Target includes main system
        if any(uuid in external_boundary_uuids for uuid in source_host_uuids) and \
           any(uuid == main_system_component_uuid for uuid in target_host_uuids):
            is_external_to_system = True
        # Condition 2: Source includes main system, Target is external
        elif any(uuid == main_system_component_uuid for uuid in source_host_uuids) and \
             any(uuid in external_boundary_uuids for uuid in target_host_uuids):
            is_external_to_system = True

        if is_external_to_system:
            external_functional_exchanges.append(fe)

# 6. Print the total number of identified external functional exchanges and their names
print(f"\nTotal number of external functional exchanges identified: {len(external_functional_exchanges)}")
if external_functional_exchanges:
    print("External Functional Exchanges:")
    for fe in external_functional_exchanges:
        # Re-call helper to get names for printing, ensuring consistency
        source_host_names = ", ".join(getattr(model.by_uuid(uuid), 'name', 'Unknown') for uuid in get_allocated_uuids_for_functional_element(fe.source))
        target_host_names = ", ".join(getattr(model.by_uuid(uuid), 'name', 'Unknown') for uuid in get_allocated_uuids_for_functional_element(fe.target))
        print(f"- {fe.name} (Source Hosts: [{source_host_names}], Target Hosts: [{target_host_names}])")
else:
    print("  (No external functional exchanges found.)")


--- Re-identifying External Functional Exchanges ---
Main 'System' component UUID: 1df2dbd8-6c62-448f-9a11-94d870e79e55
Number of identified external boundary elements: 13


C:\Users\Boody\AppData\Local\Temp\ipykernel_21128\3925891649.py:16: DeprecationWarning: str-based xsi:type handling is deprecated, use 'helpers.qtype_of(elem)' and 'etree.QName' instead
  if hasattr(functional_element, 'xtype') and \
C:\Users\Boody\AppData\Local\Temp\ipykernel_21128\3925891649.py:17: DeprecationWarning: str-based xsi:type handling is deprecated, use 'helpers.qtype_of(elem)' and 'etree.QName' instead
  ('FunctionOutputPort' in functional_element.xtype or 'FunctionInputPort' in functional_element.xtype):
C:\Users\Boody\AppData\Local\Temp\ipykernel_21128\3925891649.py:18: FutureWarning: FunctionPort.owner is deprecated, use parent instead
  current_func_or_port = functional_element.owner
C:\Users\Boody\AppData\Local\Temp\ipykernel_21128\3925891649.py:30: DeprecationWarning: str-based xsi:type handling is deprecated, use 'helpers.qtype_of(elem)' and 'etree.QName' instead
  xtype = getattr(current_element_in_traversal, 'xtype', '')



Total number of external functional exchanges identified: 19
External Functional Exchanges:
- FE RF Input
 (Source Hosts: [RX Resources (ext)], Target Hosts: [System])
- FE Streaming Data
 (Source Hosts: [System], Target Hosts: [SDR Customer Organization])
- FE RX Tasking
 (Source Hosts: [SDR Customer Organization], Target Hosts: [System])
- FE RX Task Display
 (Source Hosts: [System], Target Hosts: [SDR Operator])
- FE Detection Data
 (Source Hosts: [System], Target Hosts: [SDR Customer Organization])
- FE Detection Products
 (Source Hosts: [System], Target Hosts: [SDR Operator])
- FE Status Metrics
 (Source Hosts: [System], Target Hosts: [SDR Operator])
- FE SDR Status (Source Hosts: [System], Target Hosts: [SDR Operator])
- FE Status Display (Source Hosts: [System], Target Hosts: [SDR Operator])
- FE TX Tasking (Source Hosts: [SDR Customer Organization], Target Hosts: [System])
- TX Signal Request (Source Hosts: [TX Sources (ext)], Target Hosts: [System])
- TX RF Output (Source Hos

In [29]:
attrs = [a for a in dir(model.sa) if not a.startswith("_")]
for a in attrs:
    print(a)

all_actor_exchanges
all_actors
all_capabilities
all_capability_exploitations
all_classes
all_collections
all_complex_values
all_component_exchanges
all_components
all_enumerations
all_function_exchanges
all_functional_chains
all_functions
all_interfaces
all_missions
all_module_types
all_relation_types
all_requirement_types
all_requirements
all_unions
applied_property_value_groups
applied_property_values
capability_pkg
component_exchange_categories
component_exchange_realizations
component_exchanges
component_pkg
constraints
data_pkg
description
diagrams
enumeration_property_types
extensions
features
filtering_criteria
function_pkg
functional_allocations
functional_links
interface_pkg
is_visible_in_doc
is_visible_in_lm
iter_ancestors
layer
migrated_elements
mission_pkg
name
naming_rules
operational_analysis_realizations
parent
progress_status
property_value_groups
property_value_pkgs
property_values
pvmt
realized_operational_analysis
requirement_modules
requirement_types_folders
require

# Clean Requirement Names Before Generating Statements

In [30]:
import re

def clean_exchange_name(name):
    if not name:
        return "data"

    # remove FE prefix
    name = re.sub(r"^FE\s*", "", name)

    # remove ugly autogenerated Capella names
    if "FunctionalExchange" in name:
        return "maintenance data"

    # trim whitespace
    name = name.strip()

    return name

# Generate Requirement Statements from External Functional Exchanges

In [31]:
requirements = []

for i, fe in enumerate(external_functional_exchanges, start=1):

    clean_name = clean_exchange_name(fe.name)

    source = fe.source.owner.name if fe.source and fe.source.owner else "external system"
    target = fe.target.owner.name if fe.target and fe.target.owner else "external system"

    if main_system_component_uuid in target_host_uuids:
        statement = f"The system shall receive {clean_name} from {source}."

    elif main_system_component_uuid in source_host_uuids:
        statement = f"The system shall provide {clean_name} to {target}."

    else:
        statement = f"The system shall exchange {clean_name} with external systems."

    requirements.append({
        "id": f"REQ-SA-EXT-FE-{i:03}",
        "statement": statement
    })

C:\Users\Boody\AppData\Local\Temp\ipykernel_21128\151945937.py:7: FutureWarning: FunctionPort.owner is deprecated, use parent instead
  source = fe.source.owner.name if fe.source and fe.source.owner else "external system"
C:\Users\Boody\AppData\Local\Temp\ipykernel_21128\151945937.py:8: FutureWarning: FunctionPort.owner is deprecated, use parent instead
  target = fe.target.owner.name if fe.target and fe.target.owner else "external system"


In [32]:
sab_diagram = None

if hasattr(model.sa, "diagrams") and model.sa.diagrams:
    for diagram in model.sa.diagrams:
        if diagram.name == "[SAB] Structure":
            sab_diagram = diagram
            break

if sab_diagram is None:
    print("Warning: [SAB] Structure diagram not found.")
else:
    print(f"Found SAB diagram: {sab_diagram.name}")

Found SAB diagram: [SAB] Structure


In [33]:
rvtm = []
for req in requirements:
    rvtm.append({
        "requirement_id": req["id"],
        "requirement_statement": req["statement"],
        "verification_method": "Test",
        "verification_artifact": "TBD",
        "success_criteria": "TBD",
        "status": "Planned",
        "notes": ""
    })

# Render the Requirements into the IRS HTML Template

In [34]:
from jinja2 import Template
from datetime import datetime

# Build the IRS data structure expected by the template
irs = {
    "document": {
        "title": "INTERFACE REQUIREMENTS SPECIFICATION",
        "subtitle": "Software-Defined Radio",
        "document_control": {
            "document_id": "SDR-IRS-001",
            "version": "1.0",
            "date": datetime.now().strftime("%d %B %Y"),
            "program": "AI4SE Capstone",
            "prepared_by": "AI4SE Team",
            "approval_authority": "TBD"
        }
    },

    "scope_and_purpose": {
        "purpose": "This document defines the interface requirements for the Software Defined Radio (SDR) system.",
        "scope": "The requirements specified herein are derived from system architecture models and define external functional exchanges across the SDR system boundary."
    },

    "sab_diagram": sab_diagram,

    "interface_identification": {
        "interface_id": "SDR-External-Interfaces",
        "source_system": "External Actors",
        "destination_system": "SDR System",
        "interface_type": "Functional Exchange",
        "boundary_description": "Defines interfaces that cross the SDR system boundary."
    },

    "interface_requirements": {
        "categories": [
            {
                "name": "External Functional Exchanges",
                "requirements": requirements
            }
        ]
    },

    "verification": [],

    "rvtm": rvtm,

    "notes_and_constraints": [],

    "approvals": {
        "signatories": []
    },

    "footer": {
        "text": "George Washington University | AI4SE Capstone | SDR-IRS-001 | Version 1.1"
    }
}

# IRS HTML Template (inline)
IRS_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{{ irs.document.title }}{% if irs.document.subtitle %} - {{ irs.document.subtitle }}{% endif %}</title>

  <style>
    :root {
      --primary-color: #2f4358;
      --secondary-color: #31465c;
      --accent-color: #4aa3df;
      --text-color: #222;
      --bg-color: #f4f5f7;
      --paper-color: #ffffff;
      --border-color: #d9dde2;
      --muted-color: #6c7a89;
    }

    body {
      font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif;
      line-height: 1.6;
      color: var(--text-color);
      background-color: var(--bg-color);
      margin: 0;
      padding: 20px;
    }

    .document-container {
      max-width: 850px;
      margin: 0 auto;
      background: var(--paper-color);
      padding: 56px 60px;
      box-shadow: 0 4px 15px rgba(0,0,0,0.08);
      border-radius: 4px;
    }

    header {
      border-bottom: 2px solid var(--primary-color);
      padding-bottom: 18px;
      margin-bottom: 36px;
      text-align: center;
    }

    h1 {
      color: var(--primary-color);
      font-size: 2rem;
      margin: 0 0 8px 0;
      text-transform: uppercase;
      letter-spacing: 0.5px;
      font-weight: 700;
    }

    .subtitle {
      font-size: 1rem;
      color: var(--muted-color);
      margin: 0;
    }

    /* Restore normal section styling */
    h2 {
      color: var(--secondary-color);
      border-left: 4px solid var(--accent-color);
      padding-left: 12px;
      margin-top: 32px;
      margin-bottom: 12px;
      font-size: 1.4rem;
      letter-spacing: 0;
      white-space: normal;
    }

    h3 {
      color: var(--secondary-color);
      font-size: 1.1rem;
      margin-top: 22px;
      margin-bottom: 10px;
    }

    p {
      margin: 10px 0 0 0;
    }

    .meta-table {
      width: 100%;
      border-collapse: collapse;
      table-layout: fixed;
      margin-bottom: 28px;
      font-size: 0.95rem;
    }

    .meta-table th,
    .meta-table td {
      border: 1px solid var(--border-color);
      padding: 10px 12px;
      text-align: left;
      vertical-align: top;
    }

    .meta-table th {
      background-color: #3a4b5c;
      color: white;
      width: 30%;
    }

    /* Restore normal IRS table sizing */
    .req-table {
      width: 100%;
      border-collapse: collapse;
      margin: 18px 0;
      font-size: 0.92rem;
      page-break-inside: auto;
    }

    .req-table th,
    .req-table td {
      border: 1px solid var(--border-color);
      padding: 10px;
      vertical-align: top;
      word-wrap: break-word;
    }

    .req-table tr {
      /* removed page break */
      page-break-after: auto;
    }

    .req-table th {
      background-color: #eef2f5;
      color: var(--secondary-color);
    }

    .req-id {
      font-family: "Courier New", Courier, monospace;
      font-weight: bold;
      color: var(--accent-color);
      word-break: break-word;
    }

    .diagram-table {
      width: 100%;
      border-collapse: collapse;
      margin: 20px 0;
    }

    .diagram-table td {
      border: 1px solid var(--border-color);
      padding: 15px;
      text-align: center;
    }

    .diagram-table caption {
      caption-side: top;
      text-align: center;
      font-style: italic;
      padding: 8px;
      color: var(--secondary-color);
    }

    svg {
      width: 100% !important;
      height: auto !important;
      max-width: 100%;
    }

    .footer {
      margin-top: 56px;
      padding-top: 18px;
      border-top: 1px solid var(--border-color);
      font-size: 0.8rem;
      color: #7f8c8d;
      text-align: center;
    }

    .signature-block {
      margin-top: 48px;
      display: flex;
      justify-content: space-between;
      gap: 20px;
    }

    .sig-line {
      border-top: 1px solid #333;
      width: 45%;
      padding-top: 5px;
      text-align: center;
      font-size: 0.9rem;
    }

    /* RVTM ONLY */
    @page rvtm_page {
      size: A4 landscape;
      margin: 1in;
    }

    .rvtm-section {
      page: rvtm_page;
      page-break-before: always;
    }

    .rvtm-section h2 {
      font-size: 1.15rem;
      word-break: break-word;
      margin-top: 0;
      margin-bottom: 10px;
    }

    .rvtm-section .req-table {
      font-size: 10px;
      margin: 12px 0;
    }

    .rvtm-section .req-table th,
    .rvtm-section .req-table td {
      padding: 6px;
    }

    @media print {
      body {
        background: white;
        padding: 0;
      }

      .document-container {
        box-shadow: none;
        padding: 0;
      }
    }
  </style>
</head>

<body>
  <div class="document-container">

    <!-- HEADER -->
    <header>
      <h1>{{ irs.document.title }}</h1>
      {% if irs.document.subtitle %}
        <p class="subtitle">{{ irs.document.subtitle }}</p>
      {% endif %}
    </header>

    <!-- DOCUMENT CONTROL -->
    <section>
      <h3>Document Control</h3>
      <table class="meta-table">
        <tr>
          <th>Document ID</th>
          <td>{{ irs.document.document_control.document_id }}</td>
        </tr>
        <tr>
          <th>Version</th>
          <td>{{ irs.document.document_control.version }}</td>
        </tr>
        <tr>
          <th>Date</th>
          <td>{{ irs.document.document_control.date }}</td>
        </tr>
        <tr>
          <th>Program</th>
          <td>{{ irs.document.document_control.program }}</td>
        </tr>
        <tr>
          <th>Prepared by</th>
          <td>{{ irs.document.document_control.prepared_by }}</td>
        </tr>
        <tr>
          <th>Approval Authority</th>
          <td>{{ irs.document.document_control.approval_authority }}</td>
        </tr>
      </table>
    </section>

    <!-- 1. INTRODUCTION -->
    <section>
      <h2>1. Introduction</h2>

      <h3>1.1 Purpose</h3>
      <p>{{ irs.scope_and_purpose.purpose }}</p>

      <h3>1.2 Scope</h3>
      <p>{{ irs.scope_and_purpose.scope }}</p>
    </section>

    <!-- 2. SYSTEM ARCHITECTURE DIAGRAM -->
    {% if irs.sab_diagram %}
    <section>
      <h2>2. System Architecture Diagram</h2>
      <p>
        The System Architecture Blank (SAB) diagram shown below provides the architectural
        context from which the external functional exchanges were derived. These exchanges
        form the basis of the interface requirements defined in this document.
      </p>

      <table class="diagram-table">
        <caption>Figure 1 — {{ irs.sab_diagram.name }}</caption>
        <tr>
          <td>
            {{ irs.sab_diagram.render("svg") | safe }}
          </td>
        </tr>
      </table>
    </section>
    {% endif %}

    <!-- 3. INTERFACE IDENTIFICATION -->
    <section>
      <h2>3. Interface Identification</h2>

      <table class="req-table">
        <thead>
          <tr>
            <th>Interface ID</th>
            <th>Source System</th>
            <th>Destination System</th>
            <th>Interface Type</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td class="req-id">{{ irs.interface_identification.interface_id }}</td>
            <td>{{ irs.interface_identification.source_system }}</td>
            <td>{{ irs.interface_identification.destination_system }}</td>
            <td>{{ irs.interface_identification.interface_type }}</td>
          </tr>
        </tbody>
      </table>

      <p>
        <strong>Boundary Description:</strong>
        {{ irs.interface_identification.boundary_description }}
      </p>
    </section>

    <!-- 4. INTERFACE REQUIREMENTS -->
    <section>
      <h2>4. Interface Requirements</h2>

      {% if irs.interface_requirements and irs.interface_requirements.categories %}
        {% for cat in irs.interface_requirements.categories %}
          <h3>4.{{ loop.index }} {{ cat.name }}</h3>

          {% if cat.requirements and (cat.requirements|length > 0) %}
            <table class="req-table">
              <thead>
                <tr>
                  <th width="20%">Requirement ID</th>
                  <th>Requirement Statement</th>
                </tr>
              </thead>
              <tbody>
                {% for req in cat.requirements %}
                  <tr>
                    <td class="req-id">{{ req.id }}</td>
                    <td>{{ req.statement }}</td>
                  </tr>
                {% endfor %}
              </tbody>
            </table>
          {% else %}
            <table class="req-table">
              <tbody>
                <tr>
                  <td class="req-id">TBD</td>
                  <td>TBD</td>
                </tr>
              </tbody>
            </table>
          {% endif %}
        {% endfor %}
      {% else %}
        <table class="req-table">
          <tbody>
            <tr>
              <td class="req-id">TBD</td>
              <td>TBD</td>
            </tr>
          </tbody>
        </table>
      {% endif %}
    </section>

    <!-- 5. RVTM -->
    <section class="rvtm-section">
      <h2>5. Requirements Verification Traceability Matrix</h2>

      <table class="req-table">
        <thead>
          <tr>
            <th width="16%">Requirement ID</th>
            <th width="34%">Requirement Statement</th>
            <th width="12%">Verification Method</th>
            <th width="14%">Verification Artifact</th>
            <th width="16%">Success Criteria</th>
            <th width="8%">Status</th>
            <th width="10%">Notes</th>
          </tr>
        </thead>
        <tbody>
          {% if irs.rvtm and (irs.rvtm|length > 0) %}
            {% for row in irs.rvtm %}
              <tr>
                <td class="req-id">{{ row.requirement_id }}</td>
                <td>{{ row.requirement_statement }}</td>
                <td>{{ row.verification_method }}</td>
                <td>{{ row.verification_artifact }}</td>
                <td>{{ row.success_criteria }}</td>
                <td>{{ row.status if row.status else "" }}</td>
                <td>{{ row.notes if row.notes else "" }}</td>
              </tr>
            {% endfor %}
          {% else %}
            <tr>
              <td class="req-id">TBD</td>
              <td>TBD</td>
              <td>TBD</td>
              <td>TBD</td>
              <td>TBD</td>
              <td></td>
              <td></td>
            </tr>
          {% endif %}
        </tbody>
      </table>
    </section>

    <!-- 6. NOTES AND CONSTRAINTS -->
    <section>
      <h2>6. Notes and Constraints</h2>

      <ol>
        {% if irs.notes_and_constraints and (irs.notes_and_constraints|length > 0) %}
          {% for n in irs.notes_and_constraints %}
            <li>
              <strong>{{ n.title }}:</strong> {{ n.description }}
            </li>
          {% endfor %}
        {% else %}
          <li><strong>TBD:</strong> TBD</li>
        {% endif %}
      </ol>
    </section>

    <!-- SIGNATURES / APPROVALS -->
    <section class="signature-block">
      {% if irs.approvals and irs.approvals.signatories %}
        {% for s in irs.approvals.signatories %}
          <div class="sig-line">
            {{ s.role }}<br />
            __________________<br />
            Date: __________
          </div>
        {% endfor %}
      {% else %}
        <div class="sig-line">
          Systems Engineering Lead<br />
          __________________<br />
          Date: __________
        </div>
        <div class="sig-line">
          Program Manager<br />
          __________________<br />
          Date: __________
        </div>
      {% endif %}
    </section>

    <!-- FOOTER -->
    <div class="footer">
      <p>{{ irs.footer.text }}</p>
    </div>

  </div>
</body>
</html>
"""

template = Template(IRS_TEMPLATE)
# Render HTML
rendered_html = template.render(irs=irs)

# Save HTML
with open("../outputs/IRS_rendered.html", "w", encoding="utf-8") as f:
    f.write(rendered_html)

print("IRS document rendered successfully.")

IRS document rendered successfully.


# Export the IRS HTML to PDF

In [35]:
from weasyprint import HTML

HTML("../outputs/IRS_rendered.html").write_pdf("../outputs/IRS_rendered.pdf")

print("IRS PDF generated successfully.")

IRS PDF generated successfully.


# LLM Setup — Gemma 4 via Google Gemini API

In [36]:
!pip install openai google-generativeai python-dotenv

In [37]:
import os, json
from dotenv import load_dotenv, find_dotenv
import google.generativeai as genai

load_dotenv(find_dotenv())

# Configure the native Google Generative AI library
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Using Gemini 2.5 Flash Lite for high-speed processing
MODEL_ID = "gemini-2.5-flash-lite"
model = genai.GenerativeModel(MODEL_ID)

print(f"Google AI Studio (Native) initialized. Model: {MODEL_ID}")

Google AI Studio (Native) initialized. Model: gemini-2.5-flash-lite


# LLM Step 1 — Assign Verification Methods & Group Requirements

In [38]:
import time, json
# ── Build the prompt for verification method assignment and requirement grouping ──

req_methods = {}
req_groups = {}
group_info = {}

requirements_text = "\n".join([
    f"  {req['id']}: {req['statement']}"
    for req in requirements
])

grouping_prompt = f"""You are a senior DoD systems engineer with expertise in test and evaluation (T&E) planning for defense acquisition programs.

You are given a set of interface requirements from an Interface Requirements Specification (IRS) for a Software-Defined Radio (SDR) system. Perform the following two tasks:

**TASK 1 — Verification Method Assignment:**
For each requirement, assign the single most appropriate verification method:
- **T (Test):** Operate the system under controlled conditions; record and evaluate quantitative results.
- **A (Analysis):** Use modeling, simulation, or calculation to verify compliance.
- **I (Inspection):** Visual examination of design, documentation, or hardware without operating the system.
- **D (Demonstration):** Operate the system to qualitatively show a capability exists.

Most interface requirements will be verified by Test (T), but use your engineering judgment.

**TASK 2 — Requirement Grouping:**
Group requirements that should be tested together in a single test procedure. Requirements should be grouped if they share:
- Similar test environment or equipment setup
- Same verification method
- Related interface type or data flow direction (e.g., all \"receive\" interfaces from the same external actor)
- Logical coupling (testing one naturally tests the other)

Aim for 3-6 logical test procedure groups total. Each requirement must belong to exactly one group.

**REQUIREMENTS:**
{requirements_text}

**CRITICAL INSTRUCTION:**
You MUST distribute the requirements across at least 3 to 6 distinct groups. DO NOT put all 19 requirements into a single group (e.g., 'Group 1'). Split them logically by external actor, interface type, or data flow direction.

**RESPOND WITH VALID JSON ONLY — no markdown, no commentary:**
{{
  "requirements": [
    {{"id": "REQ-001", "verification_method": "T", "group_id": 1}},
    {{"id": "REQ-002", "verification_method": "T", "group_id": 2}},
    {{"id": "REQ-003", "verification_method": "T", "group_id": 3}}
  ],
  "groups": [
    {{
      "group_id": 1,
      "title": "Title for Group 1 (e.g., Data Reception)",
      "rationale": "Explanation"
    }},
    {{
      "group_id": 2,
      "title": "Title for Group 2 (e.g., Status Reporting)",
      "rationale": "Explanation"
    }},
    {{
      "group_id": 3,
      "title": "Title for Group 3 (e.g., Command Execution)",
      "rationale": "Explanation"
    }}
  ]
}}"""

print(f"Sending {len(requirements)} requirements to Gemini for unified grouping and method assignment...")
    

import time

max_retries = 3
for attempt in range(max_retries):
    try:
        response = model.generate_content(
            grouping_prompt,
            generation_config={
                "response_mime_type": "application/json",
                "temperature": 0.4,
            }
        )
        grouping_result = json.loads(response.text)
        
        for g in grouping_result.get("groups", []):
            group_info[g["group_id"]] = g
            
        for r in grouping_result.get("requirements", []):
            req_methods[r["id"]] = r["verification_method"]
            req_groups[r["id"]] = r["group_id"]
            
        break # Success, exit retry loop
    except Exception as e:
        print(f"  Error processing grouping (Attempt {attempt + 1}/{max_retries}): {e}")
        if attempt < max_retries - 1:
            print("  Waiting 60 seconds before retrying...")
            time.sleep(60)
        else:
            print("  Failed after maximum retries. Requirements will default to Group 1.")
print(f"\nVerification methods assigned for {len(req_methods)} requirements.")
print(f"{len(group_info)} total groups created:")
for gid, info in group_info.items():
    member_count = sum(1 for v in req_groups.values() if v == gid)
    print(f"  Group {gid}: {info['title']} ({member_count} requirements)")


Sending 19 requirements to Gemini for unified grouping and method assignment...

Verification methods assigned for 19 requirements.
5 total groups created:
  Group 1: RF and Signal Data Reception (4 requirements)
  Group 2: Tasking and Command Reception (5 requirements)
  Group 3: Status and Reporting Data Reception (6 requirements)
  Group 4: TX Signal Generation Inputs (2 requirements)
  Group 5: Waveform Package and Standards Reception (2 requirements)


# LLM Step 2 — Generate Test Plan Content

In [39]:
# ── Generate test plan body text (Static Boilerplate) ──

print("Applying static boilerplate for Test Plan body...")

tp_content = {
    "purpose": "This Test Plan defines the verification and validation strategy for the Software-Defined Radio (SDR) interface requirements. It establishes the framework for ensuring that all interfaces comply with the specifications detailed in the parent IRS document.",
    "scope": "This document covers the formal testing of all hardware and software interfaces defined in the IRS. It applies to all integration, verification, and qualification testing phases.",
    "limitations": "This plan does not cover internal component-level testing, environmental stress screening, or cyber vulnerability assessments unless explicitly linked to an IRS requirement.",
    "test_approach_overview": "Verification will follow a structured, phased approach. Interfaces will first be verified via Analysis and Inspection during design reviews. Demonstration and Test methods will be executed using simulated end-nodes in a controlled laboratory environment prior to final system integration.",
    "test_objectives": [
        "Verify all mechanical and electrical interface compliance.",
        "Validate data link protocol and message formatting accuracy.",
        "Ensure interface timing and latency requirements are met under nominal loads."
    ],
    "risks": [
        "Unavailability of external interfacing systems for live testing.",
        "Incomplete simulator fidelity for complex RF interfaces."
    ],
    "assumptions": [
        "All test equipment and simulators will be calibrated and certified prior to test execution.",
        "Software builds under test will be under formal configuration management."
    ]
}

print("Test plan content generated:")
print(f"  Purpose: {tp_content['purpose'][:80]}...")
print(f"  Objectives: {len(tp_content['test_objectives'])} items")
print(f"  Risks: {len(tp_content['risks'])} items")
print(f"  Assumptions: {len(tp_content['assumptions'])} items")


Applying static boilerplate for Test Plan body...
Test plan content generated:
  Purpose: This Test Plan defines the verification and validation strategy for the Software...
  Objectives: 3 items
  Risks: 2 items
  Assumptions: 2 items


# Build Test Plan Data & Render

In [40]:
from datetime import datetime

# ── Build test_plan data dictionary (LLM-enhanced) ──

irs_doc_id = irs["document"]["document_control"]["document_id"]
irs_doc_version = irs["document"]["document_control"]["version"]
irs_subtitle = irs["document"].get("subtitle", "Software-Defined Radio")

test_plan = {
    "document": {
        "title": "TEST PLAN",
        "subtitle": irs_subtitle,
        "document_control": {
            "document_id": "SDR-TP-001",
            "version": "1.0",
            "date": datetime.now().strftime("%d %B %Y"),
            "program": irs["document"]["document_control"]["program"],
            "prepared_by": irs["document"]["document_control"]["prepared_by"],
            "approval_authority": irs["document"]["document_control"]["approval_authority"],
            "distribution_statement": "Distribution Statement A: Approved for public release; distribution is unlimited.",
            "revision_history": [{
                "version": "1.0",
                "date": datetime.now().strftime("%d %B %Y"),
                "author": irs["document"]["document_control"]["prepared_by"],
                "changes": "Initial release"
            }]
        }
    },

    "scope_and_purpose": {
        "purpose": tp_content["purpose"],
        "scope": tp_content["scope"],
        "limitations": tp_content.get("limitations", "")
    },

    "referenced_documents": [
        {
            "document_id": irs_doc_id,
            "title": f"Interface Requirements Specification — {irs_subtitle}",
            "version": irs_doc_version
        }
    ],

    "test_objectives": tp_content["test_objectives"],

    "test_items": [
        {
            "name": "SDR External Interfaces",
            "description": f"The set of all {len(requirements)} external functional exchanges crossing the SDR system boundary, including data inputs, data outputs, command interfaces, and status reporting interfaces."
        }
    ],

    "requirements_under_test": {
        "categories": [
            {
                "name": cat["name"],
                "requirements": [
                    {
                        "id": req["id"],
                        "statement": req["statement"],
                        "verification_method": req_methods.get(req["id"], "Test")
                    }
                    for req in cat["requirements"]
                ]
            }
            for cat in irs["interface_requirements"]["categories"]
        ]
    },

    "test_approach": {
        "overview": tp_content["test_approach_overview"],
        "test_levels": "Interface-level testing; system boundary verification",
        "test_methods": "Test (T), Analysis (A), Inspection (I), Demonstration (D)",
        "pass_fail_basis": "Each requirement is evaluated individually against its specified acceptance criteria. A requirement passes if the observed behavior matches the expected behavior as defined in the requirement statement and success criteria.",
        "regression_strategy": "Any requirement that fails verification shall be retested after corrective action. Regression testing shall include the failed requirement and any related requirements that may be affected by the change."
    },

    "test_environment": {
        "location": "TBD",
        "simulation_assets": "TBD",
        "software_tools": "TBD",
        "equipment": "TBD",
        "data_requirements": "TBD",
        "security_considerations": None,
        "configuration_management": None
    },

    "entry_exit_criteria": {
        "entry": [
            f"Parent IRS ({irs_doc_id}) has been reviewed and baselined.",
            "Test procedures have been developed, reviewed, and approved.",
            "Test environment has been configured and validated.",
            "System Under Test (SUT) has been delivered and is available for testing.",
            "All required test equipment and tools are available and calibrated."
        ],
        "exit": [
            "All planned test procedures have been executed.",
            "All requirements have been verified with a Pass, Fail, or Waived disposition.",
            "All Critical and Major defects have been resolved or have approved waivers.",
            "The RVTM has been updated with actual verification results.",
            "The Test Report has been drafted and submitted for review."
        ],
        "suspension": [
            "A Critical defect is discovered that prevents further meaningful test execution.",
            "A safety concern arises that could endanger personnel or equipment.",
            "The test environment becomes unavailable, corrupted, or no longer representative.",
            "The SUT configuration is found to be incorrect or out of baseline."
        ],
        "resumption": [
            "The condition that triggered suspension has been resolved and verified.",
            "The test environment has been restored to a known-good configuration.",
            "The SUT has been returned to its baselined configuration.",
            "The Test Lead has authorized resumption in writing."
        ]
    },

    "roles_and_responsibilities": [
        {"role": "Test Lead", "responsibility": "Overall responsibility for test planning, execution, and reporting. Authorizes test start, suspension, and resumption."},
        {"role": "Test Engineer", "responsibility": "Executes test procedures, records results, and documents defects."},
        {"role": "Systems Engineer", "responsibility": "Provides technical guidance on requirement interpretation and acceptance criteria. Reviews test results."},
        {"role": "Configuration Manager", "responsibility": "Maintains configuration baselines for the SUT and test environment."},
        {"role": "Quality Assurance", "responsibility": "Witnesses critical tests, audits test records, and verifies compliance with the test plan."}
    ],

    "schedule": [],

    "risks_and_assumptions": {
        "risks": tp_content["risks"],
        "assumptions": tp_content["assumptions"]
    },

    "defect_management": {
        "process": "All test failures, anomalies, and deviations from expected results shall be documented as defect reports. Each defect report shall include: the procedure step at which the failure occurred, the requirement ID under test, observed versus expected behavior, severity classification, and any supporting evidence.",
        "tools": "TBD",
        "severity_levels": "Critical — prevents mission-essential functionality; Major — degrades performance or capability; Minor — cosmetic or low-impact deviation; Informational — observation with no functional impact",
        "disposition_authority": "Test Lead (Minor/Informational); Systems Engineering Lead (Major/Critical)"
    },

    "traceability": [
        {
            "requirement_id": req["id"],
            "test_id": f"TP-{req['id'].replace('REQ-', '')}",
            "verification_method": req_methods.get(req["id"], "Test"),
            "objective": f"Verify: {req['statement'][:80]}{'...' if len(req['statement']) > 80 else ''}"
        }
        for req in requirements
    ],

    "test_deliverables": [
        {"deliverable": "Test Plan", "description": "This document; defines the overall test strategy, scope, and schedule.", "due": "TBD"},
        {"deliverable": "Test Procedures", "description": "Step-by-step procedures for executing each planned test case.", "due": "TBD"},
        {"deliverable": "Test Report", "description": "Summary of test results, defects encountered, and verification status.", "due": "TBD"},
        {"deliverable": "Updated RVTM", "description": "Requirements Verification Traceability Matrix updated with actual test results.", "due": "TBD"}
    ],

    "notes_and_constraints": [],
    "approvals": {"signatories": []},
    "footer": {
        "text": "George Washington University | AI4SE Capstone | SDR-TP-001 | Version 1.0"
    }
}

print(f"Test plan data built: {len(test_plan['traceability'])} requirements traced, referencing parent IRS {irs_doc_id}")

Test plan data built: 19 requirements traced, referencing parent IRS SDR-IRS-001


In [41]:
from jinja2 import Template

TEST_PLAN_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{{ test_plan.document.title }}{% if test_plan.document.subtitle %} - {{ test_plan.document.subtitle }}{% endif %}</title>

  <style>
    :root {
      --primary-color: #2c3e50;
      --secondary-color: #34495e;
      --accent-color: #3498db;
      --text-color: #333;
      --bg-color: #f4f7f6;
      --paper-color: #ffffff;
      --border-color: #ddd;
    }

    body {
      font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
      line-height: 1.6;
      color: var(--text-color);
      background-color: var(--bg-color);
      margin: 0;
      padding: 20px;
    }

    .document-container {
      max-width: 900px;
      margin: 0 auto;
      background: var(--paper-color);
      padding: 60px;
      box-shadow: 0 4px 15px rgba(0,0,0,0.1);
      border-radius: 4px;
    }

    header {
      border-bottom: 2px solid var(--primary-color);
      padding-bottom: 20px;
      margin-bottom: 40px;
      text-align: center;
    }

    h1 {
      color: var(--primary-color);
      font-size: 2.2rem;
      margin-bottom: 10px;
      text-transform: uppercase;
      letter-spacing: 1px;
    }

    h2 {
      color: var(--secondary-color);
      border-left: 5px solid var(--accent-color);
      padding-left: 15px;
      margin-top: 40px;
      font-size: 1.5rem;
    }

    h3 {
      color: var(--secondary-color);
      font-size: 1.2rem;
      margin-top: 25px;
    }

    h4 {
      color: var(--secondary-color);
      font-size: 1.05rem;
      margin-top: 20px;
    }

    p, li {
      font-size: 0.98rem;
    }

    .meta-table,
    .std-table {
      width: 100%;
      border-collapse: collapse;\n",
        "      table-layout: fixed;
      table-layout: fixed;
      margin: 20px 0;
      font-size: 0.92rem;
      page-break-inside: auto;
    }

    .meta-table th, .meta-table td,
    .std-table th, .std-table td {
      border: 1px solid var(--border-color);
      padding: 10px 12px;
      text-align: left;
      vertical-align: top;
      word-wrap: break-word;
    }

    .meta-table th {
      background-color: var(--primary-color);
      color: white;
      width: 30%;
    }

    .std-table th {
      background-color: #ecf0f1;
      color: var(--secondary-color);
    }

    .std-table tr {
      /* removed page break */
      page-break-after: auto;
    }

    .id-cell {
      word-break: break-word;
      font-family: 'Courier New', Courier, monospace;
      font-weight: bold;
      color: var(--accent-color);
      word-break: break-word;
    }

    .footer {
      margin-top: 60px;
      padding-top: 20px;
      border-top: 1px solid var(--border-color);
      font-size: 0.8rem;
      color: #7f8c8d;
      text-align: center;
    }

    .signature-block {
      margin-top: 50px;
      display: flex;
      justify-content: space-between;
      gap: 20px;
    }

    .sig-line {
      border-top: 1px solid #333;
      width: 45%;
      padding-top: 5px;
      text-align: center;
      font-size: 0.9rem;
    }

    .placeholder {
      color: #7f8c8d;
      font-style: italic;
    }

    .note-box {
      border-left: 4px solid var(--accent-color);
      background: #f8fbfd;
      padding: 12px 15px;
      margin: 15px 0;
    }

    .distribution-stmt {
      text-align: center;
      font-weight: bold;
      font-size: 0.95rem;
      margin: 15px 0 30px 0;
      padding: 10px;
      border: 1px solid var(--border-color);
      background: #fefefe;
    }

    .def-table td:first-child {
      font-weight: bold;
      word-break: break-word;
      width: 20%;
    }

    @media print {
      body {
        background: white;
        padding: 0;
      }

      .document-container {
        box-shadow: none;
        padding: 0;
      }
    }
  </style>
</head>

<body>
  <div class="document-container">

    <!-- HEADER -->
    <header>
      <h1>{{ test_plan.document.title }}</h1>
      {% if test_plan.document.subtitle %}
        <p style="font-size: 1.2rem; color: #7f8c8d;">{{ test_plan.document.subtitle }}</p>
      {% endif %}
    </header>

    <!-- DOCUMENT CONTROL -->
    <section>
      <h3>Document Control</h3>
      <table class="meta-table">
        <tr>
          <th>Document ID</th>
          <td>{{ test_plan.document.document_control.document_id }}</td>
        </tr>
        <tr>
          <th>Version</th>
          <td>{{ test_plan.document.document_control.version }}</td>
        </tr>
        <tr>
          <th>Date</th>
          <td>{{ test_plan.document.document_control.date }}</td>
        </tr>
        <tr>
          <th>Program</th>
          <td>{{ test_plan.document.document_control.program }}</td>
        </tr>
        <tr>
          <th>Prepared by</th>
          <td>{{ test_plan.document.document_control.prepared_by }}</td>
        </tr>
        <tr>
          <th>Approval Authority</th>
          <td>{{ test_plan.document.document_control.approval_authority }}</td>
        </tr>
      </table>

      <!-- REVISION HISTORY -->
      <h4>Revision History</h4>
      <table class="std-table">
        <thead>
          <tr>
            <th>Version</th>
            <th>Date</th>
            <th>Author</th>
            <th>Changes</th>
          </tr>
        </thead>
        <tbody>
          {% if test_plan.document.document_control.revision_history and (test_plan.document.document_control.revision_history|length > 0) %}
            {% for rev in test_plan.document.document_control.revision_history %}
              <tr>
                <td>{{ rev.version }}</td>
                <td>{{ rev.date }}</td>
                <td>{{ rev.author }}</td>
                <td>{{ rev.changes }}</td>
              </tr>
            {% endfor %}
          {% else %}
            <tr>
              <td>{{ test_plan.document.document_control.version }}</td>
              <td>{{ test_plan.document.document_control.date }}</td>
              <td>{{ test_plan.document.document_control.prepared_by }}</td>
              <td>Initial release</td>
            </tr>
          {% endif %}
        </tbody>
      </table>
    </section>

    

    <!-- 1. SCOPE AND PURPOSE -->
    <section>
      <h2>1. Scope and Purpose</h2>
      <p><strong>Purpose:</strong> {{ test_plan.scope_and_purpose.purpose }}</p>
      <p><strong>Scope:</strong> {{ test_plan.scope_and_purpose.scope }}</p>

      {% if test_plan.scope_and_purpose.limitations %}
        <p><strong>Limitations:</strong> {{ test_plan.scope_and_purpose.limitations }}</p>
      {% endif %}
    </section>

    <!-- 2. REFERENCED DOCUMENTS -->
    <section>
      <h2>2. Referenced Documents</h2>
      <p>The following documents are referenced herein and form a part of this test plan to the extent specified.</p>

      <table class="std-table">
        <thead>
          <tr>
            <th width="25%">Document ID</th>
            <th>Title</th>
            <th width="12%">Version</th>
          </tr>
        </thead>
        <tbody>
          {% if test_plan.referenced_documents and (test_plan.referenced_documents|length > 0) %}
            {% for doc in test_plan.referenced_documents %}
              <tr>
                <td class="id-cell">{{ doc.document_id }}</td>
                <td>{{ doc.title }}</td>
                <td>{{ doc.version if doc.version else "—" }}</td>
              </tr>
            {% endfor %}
          {% else %}
            <tr>
              <td class="id-cell">TBD</td>
              <td>TBD</td>
              <td>—</td>
            </tr>
          {% endif %}
        </tbody>
      </table>
    </section>

    <!-- 3. DEFINITIONS, ACRONYMS, AND ABBREVIATIONS -->
    <section>
      <h2>3. Definitions, Acronyms, and Abbreviations</h2>

      <table class="std-table">
        <thead>
          <tr>
            <th width="25%">Term / Acronym</th>
            <th>Definition</th>
          </tr>
        </thead>
        <tbody>
          <tr><td><strong>DT</strong></td><td>Developmental Test — testing conducted to evaluate design maturity and technical performance.</td></tr>
          <tr><td><strong>IRS</strong></td><td>Interface Requirements Specification — a document defining interface requirements between systems or system elements.</td></tr>
          <tr><td><strong>OT</strong></td><td>Operational Test — testing conducted to evaluate system effectiveness and suitability in an operationally representative environment.</td></tr>
          <tr><td><strong>RVTM</strong></td><td>Requirements Verification Traceability Matrix — a matrix linking requirements to their planned verification activities and results.</td></tr>
          <tr><td><strong>SUT</strong></td><td>System Under Test — the system or system element being evaluated during testing.</td></tr>
          <tr><td><strong>T&amp;E</strong></td><td>Test and Evaluation — the process of planning, conducting, and assessing tests to provide information about system capabilities and limitations.</td></tr>
          <tr><td><strong>TAID</strong></td><td>Test, Analysis, Inspection, Demonstration — the four standard verification methods used to confirm requirement compliance.</td></tr>
          <tr><td><strong>TBD</strong></td><td>To Be Determined — content that has not yet been defined.</td></tr>
        </tbody>
      </table>
    </section>

    <!-- 4. TEST OBJECTIVES -->
    <section>
      <h2>4. Test Objectives</h2>

      {% if test_plan.test_objectives and (test_plan.test_objectives|length > 0) %}
        <ol>
          {% for obj in test_plan.test_objectives %}
            <li>{{ obj }}</li>
          {% endfor %}
        </ol>
      {% else %}
        <p class="placeholder">TBD: Test objectives will be defined based on the interface requirements and intended verification scope.</p>
      {% endif %}
    </section>

    <!-- 5. TEST ITEMS -->
    <section>
      <h2>5. Test Items</h2>

      {% if test_plan.test_items and (test_plan.test_items|length > 0) %}
        <table class="std-table">
          <thead>
            <tr>
              <th width="25%">Test Item</th>
              <th>Description</th>
            </tr>
          </thead>
          <tbody>
            {% for item in test_plan.test_items %}
              <tr>
                <td class="id-cell">{{ item.name }}</td>
                <td>{{ item.description }}</td>
              </tr>
            {% endfor %}
          </tbody>
        </table>
      {% else %}
        <p class="placeholder">TBD: Test items will be defined by the system, interface, and configuration under test.</p>
      {% endif %}
    </section>

    <!-- 6. REQUIREMENTS UNDER TEST -->
    <section>
      <h2>6. Requirements Under Test</h2>

      {% if test_plan.requirements_under_test and test_plan.requirements_under_test.categories %}
        {% for cat in test_plan.requirements_under_test.categories %}
          <h3>6.{{ loop.index }} {{ cat.name }}</h3>

          {% if cat.requirements and (cat.requirements|length > 0) %}
            <table class="std-table">
              <thead>
                <tr>
                  <th width="20%">Requirement ID</th>
                  <th>Requirement Statement</th>
                  <th width="18%">Planned Verification Method</th>
                </tr>
              </thead>
              <tbody>
                {% for req in cat.requirements %}
                  <tr>
                    <td class="id-cell">{{ req.id }}</td>
                    <td>{{ req.statement }}</td>
                    <td>
                      {% if req.verification_method %}
                        {{ req.verification_method }}
                      {% else %}
                        TBD
                      {% endif %}
                    </td>
                  </tr>
                {% endfor %}
              </tbody>
            </table>
          {% else %}
            <p class="placeholder">No requirements identified for this category.</p>
          {% endif %}
        {% endfor %}
      {% else %}
        <p class="placeholder">TBD: Requirements under test will be populated from the generated IRS and related verification needs.</p>
      {% endif %}
    </section>

    <!-- 7. TEST APPROACH -->
    <section>
      <h2>7. Test Approach</h2>

      <p>
        {% if test_plan.test_approach.overview %}
          {{ test_plan.test_approach.overview }}
        {% else %}
          <span class="placeholder">TBD: The overall test approach will define how interface requirements are verified through test, analysis, inspection, demonstration, or a combination thereof.</span>
        {% endif %}
      </p>

      <table class="std-table">
        <thead>
          <tr>
            <th>Approach Element</th>
            <th>Description</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>Test Levels</td>
            <td>{{ test_plan.test_approach.test_levels if test_plan.test_approach.test_levels else "TBD" }}</td>
          </tr>
          <tr>
            <td>Test Methods</td>
            <td>{{ test_plan.test_approach.test_methods if test_plan.test_approach.test_methods else "TBD" }}</td>
          </tr>
          <tr>
            <td>Pass/Fail Basis</td>
            <td>{{ test_plan.test_approach.pass_fail_basis if test_plan.test_approach.pass_fail_basis else "TBD" }}</td>
          </tr>
          <tr>
            <td>Regression / Retest Strategy</td>
            <td>{{ test_plan.test_approach.regression_strategy if test_plan.test_approach.regression_strategy else "TBD" }}</td>
          </tr>
        </tbody>
      </table>
    </section>

    <!-- 8. TEST ENVIRONMENT -->
    <section>
      <h2>8. Test Environment</h2>

      <div class="note-box">
        Environment-specific details such as exact hardware models, bench configurations,
        or organization-specific tools may remain TBD until program-level tailoring occurs.
      </div>

      <table class="std-table">
        <thead>
          <tr>
            <th>Environment Element</th>
            <th>Description</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>Test Location</td>
            <td>{{ test_plan.test_environment.location if test_plan.test_environment.location else "TBD" }}</td>
          </tr>
          <tr>
            <td>Simulation / Emulation Assets</td>
            <td>{{ test_plan.test_environment.simulation_assets if test_plan.test_environment.simulation_assets else "TBD" }}</td>
          </tr>
          <tr>
            <td>Support Software / Tools</td>
            <td>{{ test_plan.test_environment.software_tools if test_plan.test_environment.software_tools else "TBD" }}</td>
          </tr>
          <tr>
            <td>Test Equipment</td>
            <td>{{ test_plan.test_environment.equipment if test_plan.test_environment.equipment else "TBD" }}</td>
          </tr>
          <tr>
            <td>Data / Inputs Required</td>
            <td>{{ test_plan.test_environment.data_requirements if test_plan.test_environment.data_requirements else "TBD" }}</td>
          </tr>
        </tbody>
      </table>

      <!-- 8.1 SECURITY CONSIDERATIONS -->
      <h3>8.1 Security Considerations</h3>
      <p>
        {% if test_plan.test_environment.security_considerations %}
          {{ test_plan.test_environment.security_considerations }}
        {% else %}
          <span class="placeholder">TBD: Security measures, handling requirements for classified or controlled unclassified information (CUI), facility clearance requirements, and cybersecurity considerations applicable to the test environment will be defined prior to test execution.</span>
        {% endif %}
      </p>

      <!-- 8.2 CONFIGURATION MANAGEMENT -->
      <h3>8.2 Configuration Management</h3>
      <p>
        {% if test_plan.test_environment.configuration_management %}
          {{ test_plan.test_environment.configuration_management }}
        {% else %}
          <span class="placeholder">TBD: Configuration identification and control procedures for the System Under Test (SUT) will be established prior to test execution. This includes documentation of software build identifiers, hardware serial numbers, firmware versions, and test environment baselines.</span>
        {% endif %}
      </p>
    </section>

    <!-- 9. ENTRY, EXIT, SUSPENSION, AND RESUMPTION CRITERIA -->
    <section>
      <h2>9. Entry, Exit, Suspension, and Resumption Criteria</h2>

      <h3>9.1 Entry Criteria</h3>
      {% if test_plan.entry_exit_criteria.entry and (test_plan.entry_exit_criteria.entry|length > 0) %}
        <ul>
          {% for item in test_plan.entry_exit_criteria.entry %}
            <li>{{ item }}</li>
          {% endfor %}
        </ul>
      {% else %}
        <p class="placeholder">TBD: Entry criteria will be defined prior to formal test execution.</p>
      {% endif %}

      <h3>9.2 Exit Criteria</h3>
      {% if test_plan.entry_exit_criteria.exit and (test_plan.entry_exit_criteria.exit|length > 0) %}
        <ul>
          {% for item in test_plan.entry_exit_criteria.exit %}
            <li>{{ item }}</li>
          {% endfor %}
        </ul>
      {% else %}
        <p class="placeholder">TBD: Exit criteria will be defined based on completion and disposition of planned verification activities.</p>
      {% endif %}

      <h3>9.3 Suspension Criteria</h3>
      {% if test_plan.entry_exit_criteria.suspension and (test_plan.entry_exit_criteria.suspension|length > 0) %}
        <ul>
          {% for item in test_plan.entry_exit_criteria.suspension %}
            <li>{{ item }}</li>
          {% endfor %}
        </ul>
      {% else %}
        <p class="placeholder">TBD: Testing may be suspended if a critical defect is discovered that prevents further meaningful test execution, a safety concern arises, or the test environment becomes unavailable or compromised.</p>
      {% endif %}

      <h3>9.4 Resumption Criteria</h3>
      {% if test_plan.entry_exit_criteria.resumption and (test_plan.entry_exit_criteria.resumption|length > 0) %}
        <ul>
          {% for item in test_plan.entry_exit_criteria.resumption %}
            <li>{{ item }}</li>
          {% endfor %}
        </ul>
      {% else %}
        <p class="placeholder">TBD: Testing may resume once the condition that triggered suspension has been resolved, verified, and documented, and the test environment has been restored to a known-good configuration.</p>
      {% endif %}
    </section>

    <!-- 10. ROLES AND RESPONSIBILITIES -->
    <section>
      <h2>10. Roles and Responsibilities</h2>

      {% if test_plan.roles_and_responsibilities and (test_plan.roles_and_responsibilities|length > 0) %}
        <table class="std-table">
          <thead>
            <tr>
              <th width="28%">Role</th>
              <th>Responsibility</th>
            </tr>
          </thead>
          <tbody>
            {% for role in test_plan.roles_and_responsibilities %}
              <tr>
                <td>{{ role.role }}</td>
                <td>{{ role.responsibility }}</td>
              </tr>
            {% endfor %}
          </tbody>
        </table>
      {% else %}
        <p class="placeholder">TBD: Roles and responsibilities will be assigned by the program or test organization.</p>
      {% endif %}
    </section>

    <!-- 11. SCHEDULE AND MILESTONES -->
    <section>
      <h2>11. Schedule and Milestones</h2>

      {% if test_plan.schedule and (test_plan.schedule|length > 0) %}
        <table class="std-table">
          <thead>
            <tr>
              <th>Milestone</th>
              <th>Planned Date / Window</th>
              <th>Notes</th>
            </tr>
          </thead>
          <tbody>
            {% for item in test_plan.schedule %}
              <tr>
                <td>{{ item.milestone }}</td>
                <td>{{ item.date }}</td>
                <td>{{ item.notes if item.notes else "" }}</td>
              </tr>
            {% endfor %}
          </tbody>
        </table>
      {% else %}
        <p class="placeholder">TBD: Schedule and milestones will be defined by program planning constraints.</p>
      {% endif %}
    </section>

    <!-- 12. RISKS AND ASSUMPTIONS -->
    <section>
      <h2>12. Risks and Assumptions</h2>

      <h3>12.1 Risks</h3>
      {% if test_plan.risks_and_assumptions.risks and (test_plan.risks_and_assumptions.risks|length > 0) %}
        <ul>
          {% for risk in test_plan.risks_and_assumptions.risks %}
            <li>{{ risk }}</li>
          {% endfor %}
        </ul>
      {% else %}
        <p class="placeholder">TBD: Risks will be documented as the verification approach matures.</p>
      {% endif %}

      <h3>12.2 Assumptions</h3>
      {% if test_plan.risks_and_assumptions.assumptions and (test_plan.risks_and_assumptions.assumptions|length > 0) %}
        <ul>
          {% for assumption in test_plan.risks_and_assumptions.assumptions %}
            <li>{{ assumption }}</li>
          {% endfor %}
        </ul>
      {% else %}
        <p class="placeholder">TBD: Assumptions will be refined based on the test environment and available resources.</p>
      {% endif %}
    </section>

    <!-- 13. DEFECT / NON-CONFORMANCE MANAGEMENT -->
    <section>
      <h2>13. Defect / Non-Conformance Management</h2>

      <p>
        {% if test_plan.defect_management and test_plan.defect_management.process %}
          {{ test_plan.defect_management.process }}
        {% else %}
          <span class="placeholder">TBD: The defect management process will define how test failures, anomalies, and non-conformances are recorded, categorized, tracked, dispositioned, and closed. All defects shall be documented with sufficient detail to enable root-cause analysis and corrective action.</span>
        {% endif %}
      </p>

      <table class="std-table">
        <thead>
          <tr>
            <th>Element</th>
            <th>Description</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>Tracking Tool</td>
            <td>{{ test_plan.defect_management.tools if test_plan.defect_management is defined and test_plan.defect_management.tools else "TBD" }}</td>
          </tr>
          <tr>
            <td>Severity Levels</td>
            <td>{{ test_plan.defect_management.severity_levels if test_plan.defect_management is defined and test_plan.defect_management.severity_levels else "TBD (e.g., Critical, Major, Minor, Informational)" }}</td>
          </tr>
          <tr>
            <td>Disposition Authority</td>
            <td>{{ test_plan.defect_management.disposition_authority if test_plan.defect_management is defined and test_plan.defect_management.disposition_authority else "TBD" }}</td>
          </tr>
        </tbody>
      </table>
    </section>

    <!-- 14. VERIFICATION METHOD DEFINITIONS AND TRACEABILITY MATRIX -->
    <section>
      <h2>14. Traceability Matrix</h2>

      <!-- 14.1 VERIFICATION METHOD DEFINITIONS -->
      <h3>14.1 Verification Method Definitions</h3>
      <p>The following verification methods, consistent with standard systems engineering practice (INCOSE SE Handbook; ISO/IEC/IEEE 15288), are used throughout this document and the traceability matrix below.</p>

      <table class="std-table def-table">
        <thead>
          <tr>
            <th width="20%">Method</th>
            <th>Definition</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>Test (T)</td>
            <td>Operation of the system or component under controlled conditions to verify performance against specified requirements. Inputs are provided and outputs are recorded and evaluated against predicted or expected results.</td>
          </tr>
          <tr>
            <td>Analysis (A)</td>
            <td>Use of mathematical modeling, simulation, statistical evaluation, or other technical assessment methods to verify compliance with requirements where physical testing alone is impractical or insufficient.</td>
          </tr>
          <tr>
            <td>Inspection (I)</td>
            <td>Visual examination or review of documentation, design artifacts, or physical hardware to verify conformance with specified requirements without operating the system.</td>
          </tr>
          <tr>
            <td>Demonstration (D)</td>
            <td>Operation of the system to show qualitatively that a required capability exists. Demonstration relies on observable functional performance rather than detailed quantitative measurement.</td>
          </tr>
        </tbody>
      </table>

      <!-- 14.2 REQUIREMENT-TO-TEST TRACEABILITY -->
      <h3>14.2 Requirement-to-Test Traceability</h3>

      {% if test_plan.traceability and (test_plan.traceability|length > 0) %}
        <table class="std-table">
          <thead>
            <tr>
              <th width="18%">Requirement ID</th>
              <th width="18%">Planned Test ID</th>
              <th width="14%">Verification Method</th>
              <th>Verification Objective</th>
            </tr>
          </thead>
          <tbody>
            {% for row in test_plan.traceability %}
              <tr>
                <td class="id-cell">{{ row.requirement_id }}</td>
                <td class="id-cell">{{ row.test_id if row.test_id else "TBD" }}</td>
                <td>{{ row.verification_method if row.verification_method else "TBD" }}</td>
                <td>{{ row.objective if row.objective else "TBD" }}</td>
              </tr>
            {% endfor %}
          </tbody>
        </table>
      {% else %}
        <p class="placeholder">TBD: Traceability will be established between interface requirements and planned test procedures.</p>
      {% endif %}
    </section>

    <!-- 15. TEST DELIVERABLES AND REPORTING -->
    <section>
      <h2>15. Test Deliverables and Reporting</h2>

      <p>The following deliverables shall be produced as part of the test and evaluation effort described in this plan.</p>

      {% if test_plan.test_deliverables and (test_plan.test_deliverables|length > 0) %}
        <table class="std-table">
          <thead>
            <tr>
              <th width="28%">Deliverable</th>
              <th>Description</th>
              <th width="15%">Due</th>
            </tr>
          </thead>
          <tbody>
            {% for d in test_plan.test_deliverables %}
              <tr>
                <td>{{ d.deliverable }}</td>
                <td>{{ d.description }}</td>
                <td>{{ d.due if d.due else "TBD" }}</td>
              </tr>
            {% endfor %}
          </tbody>
        </table>
      {% else %}
        <table class="std-table">
          <thead>
            <tr>
              <th width="28%">Deliverable</th>
              <th>Description</th>
              <th width="15%">Due</th>
            </tr>
          </thead>
          <tbody>
            <tr>
              <td>Test Plan</td>
              <td>This document; defines the overall test strategy, scope, and schedule.</td>
              <td>TBD</td>
            </tr>
            <tr>
              <td>Test Procedures</td>
              <td>Step-by-step procedures for executing each planned test case.</td>
              <td>TBD</td>
            </tr>
            <tr>
              <td>Test Report</td>
              <td>Summary of test results, defects encountered, and verification status.</td>
              <td>TBD</td>
            </tr>
            <tr>
              <td>Updated RVTM</td>
              <td>Requirements Verification Traceability Matrix updated with actual test results and verification status.</td>
              <td>TBD</td>
            </tr>
          </tbody>
        </table>
      {% endif %}
    </section>

    <!-- 16. NOTES AND CONSTRAINTS -->
    <section>
      <h2>16. Notes and Constraints</h2>

      <ol>
        {% if test_plan.notes_and_constraints and (test_plan.notes_and_constraints|length > 0) %}
          {% for n in test_plan.notes_and_constraints %}
            <li><strong>{{ n.title }}:</strong> {{ n.description }}</li>
          {% endfor %}
        {% else %}
          <li><strong>TBD:</strong> Program-specific constraints and notes will be added during tailoring.</li>
        {% endif %}
      </ol>
    </section>

    <!-- APPROVALS -->
    <section class="signature-block">
      {% if test_plan.approvals and test_plan.approvals.signatories %}
        {% for s in test_plan.approvals.signatories %}
          <div class="sig-line">
            {{ s.role }}<br />
            __________________<br />
            Date: __________
          </div>
        {% endfor %}
      {% else %}
        <div class="sig-line">
          Systems Engineering Lead<br />
          __________________<br />
          Date: __________
        </div>
        <div class="sig-line">
          Test Lead<br />
          __________________<br />
          Date: __________
        </div>
      {% endif %}
    </section>

    <!-- FOOTER -->
    <div class="footer">
      <p>{{ test_plan.footer.text }}</p>
    </div>

  </div>
</body>
</html>

"""

tp_template = Template(TEST_PLAN_TEMPLATE)
tp_rendered_html = tp_template.render(test_plan=test_plan)

with open("../outputs/TestPlan_rendered.html", "w", encoding="utf-8") as f:
    f.write(tp_rendered_html)

print("Test Plan HTML rendered successfully.")

Test Plan HTML rendered successfully.


In [42]:
from weasyprint import HTML

HTML("../outputs/TestPlan_rendered.html").write_pdf("../outputs/TestPlan_rendered.pdf")

print("Test Plan PDF generated successfully.")

Test Plan PDF generated successfully.


# LLM Step 3 — Generate Grouped Test Procedures

In [43]:
# ── For each group, generate procedure content (Static Boilerplate) ──

print("Applying static boilerplate for Test Procedures...")

# Organize requirements by group
groups_with_reqs = {}
for req in requirements:
    gid = req_groups.get(req["id"], 1)
    if gid not in groups_with_reqs:
        groups_with_reqs[gid] = {
            "info": group_info.get(gid, {"title": f"Group {gid}", "rationale": ""}),
            "requirements": []
        }
    groups_with_reqs[gid]["requirements"].append(req)

all_procedure_content = {}

for gid, group_data in groups_with_reqs.items():
    req_ids = [r['id'] for r in group_data["requirements"]]
    
    all_procedure_content[gid] = {
        "objective": f"Verify compliance of interface requirements: {', '.join(req_ids)}.",
        "preconditions": "1. Test environment configured per Section 3.1.\n2. System Under Test (SUT) powered on and initialized to normal operating mode.\n3. Test equipment and data loggers active.",
        "pass_fail_criteria": "The SUT successfully interfaces with the simulated external systems without error, and all captured data matches the expected parameters defined in the IRS.",
        "steps": [
            {
                "step_number": 1,
                "action": "Initialize the interface connection between the SUT and the test equipment.",
                "expected_result": "Connection is established and stable. Link status indicates 'UP'."
            },
            {
                "step_number": 2,
                "action": "Transmit nominal test data across the interface.",
                "expected_result": "Data is successfully transmitted and received without degradation or loss."
            },
            {
                "step_number": 3,
                "action": "Monitor interface for unexpected resets or protocol errors for a duration of 5 minutes.",
                "expected_result": "No resets or errors observed. Interface maintains required throughput."
            }
        ],
        "evidence": "Network captures (PCAP), interface log files, and visual confirmation of link status."
    }
    
    step_count = len(all_procedure_content[gid].get("steps", []))
    print(f"  Group {gid}: Generated {step_count} static steps")

print(f"\nAll {len(all_procedure_content)} grouped procedures generated successfully.")


Applying static boilerplate for Test Procedures...
  Group 1: Generated 3 static steps
  Group 2: Generated 3 static steps
  Group 3: Generated 3 static steps
  Group 4: Generated 3 static steps
  Group 5: Generated 3 static steps

All 5 grouped procedures generated successfully.


# Build Test Procedure Data & Render

In [44]:
from datetime import datetime

# ── Build test_procedure data dictionary (LLM-enhanced, grouped) ──

irs_doc_id = irs["document"]["document_control"]["document_id"]
tp_doc_id = test_plan["document"]["document_control"]["document_id"]
irs_subtitle = irs["document"].get("subtitle", "Software-Defined Radio")

procedures = []
traceability_rows = []

for gid, group_data in groups_with_reqs.items():
    content = all_procedure_content[gid]
    req_ids = [r["id"] for r in group_data["requirements"]]
    proc_id = f"TP-GRP-{gid:03d}"

    procedures.append({
        "title": group_data["info"]["title"],
        "procedure_id": proc_id,
        "requirement_ids": req_ids,
        "objective": content["objective"],
        "verification_method": req_methods.get(req_ids[0], "Test"),
        "preconditions": content["preconditions"],
        "pass_fail_criteria": content["pass_fail_criteria"],
        "steps": content.get("steps", []),
        "evidence": content.get("evidence", "TBD"),
        "notes": group_data["info"]["rationale"]
    })

    for rid in req_ids:
        traceability_rows.append({
            "requirement_id": rid,
            "procedure_id": proc_id,
            "verification_method": req_methods.get(rid, "Test")
        })

test_procedure = {
    "document": {
        "title": "TEST PROCEDURES",
        "subtitle": irs_subtitle,
        "document_control": {
            "document_id": "SDR-TPR-001",
            "version": "1.0",
            "date": datetime.now().strftime("%d %B %Y"),
            "program": irs["document"]["document_control"]["program"],
            "prepared_by": irs["document"]["document_control"]["prepared_by"],
            "approval_authority": irs["document"]["document_control"]["approval_authority"],
            "distribution_statement": "Distribution Statement A: Approved for public release; distribution is unlimited.",
            "revision_history": [{
                "version": "1.0",
                "date": datetime.now().strftime("%d %B %Y"),
                "author": irs["document"]["document_control"]["prepared_by"],
                "changes": "Initial release"
            }]
        }
    },

    "introduction": {
        "purpose": f"This document provides the detailed test procedures for verifying the interface requirements specified in {irs_doc_id}, as planned in {tp_doc_id}. Requirements have been grouped into {len(procedures)} test procedures based on similarity of test environment, equipment, and verification method.",
        "scope": f"These procedures cover all {len(requirements)} interface requirements identified in the parent IRS. Requirements are organized into grouped procedures where related requirements are verified together for efficiency and operational realism."
    },

    "referenced_documents": [
        {
            "document_id": irs_doc_id,
            "title": f"Interface Requirements Specification — {irs_subtitle}",
            "version": irs["document"]["document_control"]["version"]
        },
        {
            "document_id": tp_doc_id,
            "title": f"Test Plan — {irs_subtitle}",
            "version": test_plan["document"]["document_control"]["version"]
        }
    ],

    "general_conditions": {
        "test_environment": "TBD — The test environment shall replicate or simulate all external interfaces as defined in the parent IRS.",
        "equipment": "TBD — Required equipment includes interface simulators, protocol analyzers, and data capture tools.",
        "initial_configuration": "The SUT shall be powered on, initialized to its default operational state, and connected to all external interface simulators before procedure execution begins.",
        "assumptions": "All external interface simulators are available and configured to generate valid stimuli. The SUT is in a known-good baseline configuration.",
        "security_considerations": None,
        "configuration_management": None
    },

    "procedures": procedures,

    "traceability": traceability_rows,

    "defect_management": {
        "process": "Any test failures, anomalies, or deviations from expected results encountered during procedure execution shall be documented as defect reports. Each report shall include the procedure ID, step number, requirement ID(s) affected, observed vs. expected behavior, severity, and supporting evidence."
    },

    "notes_and_constraints": [],
    "approvals": {"signatories": []},
    "footer": {
        "text": "George Washington University | AI4SE Capstone | SDR-TPR-001 | Version 1.0"
    }
}

print(f"Test procedure data built: {len(procedures)} grouped procedures covering {len(requirements)} requirements")
for proc in procedures:
    print(f"  {proc['procedure_id']}: {proc['title']} ({len(proc['requirement_ids'])} reqs, {len(proc['steps'])} steps)")

Test procedure data built: 5 grouped procedures covering 19 requirements
  TP-GRP-001: RF and Signal Data Reception (4 reqs, 3 steps)
  TP-GRP-002: Tasking and Command Reception (5 reqs, 3 steps)
  TP-GRP-003: Status and Reporting Data Reception (6 reqs, 3 steps)
  TP-GRP-004: TX Signal Generation Inputs (2 reqs, 3 steps)
  TP-GRP-005: Waveform Package and Standards Reception (2 reqs, 3 steps)


In [45]:
from jinja2 import Template

TEST_PROCEDURE_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{{ test_procedure.document.title }}{% if test_procedure.document.subtitle %} - {{ test_procedure.document.subtitle }}{% endif %}</title>

  <style>
    :root {
      --primary-color: #2f4358;
      --secondary-color: #31465c;
      --accent-color: #4aa3df;
      --text-color: #222;
      --bg-color: #f4f5f7;
      --paper-color: #ffffff;
      --border-color: #d9dde2;
      --muted-color: #6c7a89;
    }

    body {
      font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif;
      line-height: 1.6;
      color: var(--text-color);
      background-color: var(--bg-color);
      margin: 0;
      padding: 20px;
    }

    .document-container {
      max-width: 900px;
      margin: 0 auto;
      background: var(--paper-color);
      padding: 56px 60px;
      box-shadow: 0 4px 15px rgba(0,0,0,0.08);
      border-radius: 4px;
    }

    header {
      border-bottom: 2px solid var(--primary-color);
      padding-bottom: 18px;
      margin-bottom: 36px;
      text-align: center;
    }

    h1 {
      color: var(--primary-color);
      font-size: 2rem;
      margin: 0 0 8px 0;
      text-transform: uppercase;
      letter-spacing: 0.5px;
      font-weight: 700;
    }

    .subtitle {
      font-size: 1rem;
      color: var(--muted-color);
      margin: 0;
    }

    h2 {
      color: var(--secondary-color);
      border-left: 4px solid var(--accent-color);
      padding-left: 12px;
      margin-top: 32px;
      margin-bottom: 12px;
      font-size: 1.4rem;
    }

    h3 {
      color: var(--secondary-color);
      font-size: 1.1rem;
      margin-top: 22px;
      margin-bottom: 10px;
    }

    h4 {
      color: var(--secondary-color);
      font-size: 1rem;
      margin-top: 18px;
      margin-bottom: 8px;
    }

    p {
      margin: 10px 0 0 0;
    }

    .meta-table,
    .std-table {
      width: 100%;
      border-collapse: collapse;
      table-layout: fixed;
      margin-bottom: 24px;
      font-size: 0.95rem;
    }

    .meta-table th, .meta-table td,
    .std-table th, .std-table td {
      border: 1px solid var(--border-color);
      padding: 10px 12px;
      text-align: left;
      vertical-align: top;
      word-wrap: break-word;
    }

    .meta-table th {
      background-color: #3a4b5c;
      color: white;
      width: 30%;
    }

    .std-table th {
      background-color: #eef2f5;
      color: var(--secondary-color);
    }

    .proc-id {
      font-family: "Courier New", Courier, monospace;
      font-weight: bold;
      color: var(--accent-color);
      word-break: break-word;
    }

    .procedure-block {
      margin-top: 30px;
      padding-top: 10px;
      border-top: 1px solid var(--border-color);
      /* removed page break */
    }

    .step-table {
      width: 100%;
      border-collapse: collapse;
      margin: 15px 0 20px 0;
      font-size: 0.93rem;
    }

    .step-table th,
    .step-table td {
      border: 1px solid var(--border-color);
      padding: 10px;
      text-align: left;
      vertical-align: top;
    }

    .step-table th {
      background-color: #eef2f5;
      color: var(--secondary-color);
    }

    .step-number {
      width: 8%;
      word-break: break-word;
      font-weight: bold;
    }

    .note-box {
      border-left: 4px solid var(--accent-color);
      background: #f8fbfd;
      padding: 12px 15px;
      margin: 12px 0 18px 0;
    }

    .placeholder {
      color: var(--muted-color);
      font-style: italic;
    }

    .distribution-stmt {
      text-align: center;
      font-weight: bold;
      font-size: 0.95rem;
      margin: 15px 0 30px 0;
      padding: 10px;
      border: 1px solid var(--border-color);
      background: #fefefe;
    }

    .def-table td:first-child {
      font-weight: bold;
      word-break: break-word;
      width: 20%;
    }

    .footer {
      margin-top: 56px;
      padding-top: 18px;
      border-top: 1px solid var(--border-color);
      font-size: 0.8rem;
      color: #7f8c8d;
      text-align: center;
    }

    .signature-block {
      margin-top: 48px;
      display: flex;
      justify-content: space-between;
      gap: 20px;
    }

    .sig-line {
      border-top: 1px solid #333;
      width: 45%;
      padding-top: 5px;
      text-align: center;
      font-size: 0.9rem;
    }


    @media print {
      body {
        background: white;
        padding: 0;
      }

      .document-container {
        box-shadow: none;
        padding: 0;
      }
    }
  </style>
</head>

<body>
  <div class="document-container">

    <!-- HEADER -->
    <header>
      <h1>{{ test_procedure.document.title }}</h1>
      {% if test_procedure.document.subtitle %}
        <p class="subtitle">{{ test_procedure.document.subtitle }}</p>
      {% endif %}
    </header>

    <!-- DOCUMENT CONTROL -->
    <section>
      <h3>Document Control</h3>
      <table class="meta-table">
        <tr>
          <th>Document ID</th>
          <td>{{ test_procedure.document.document_control.document_id }}</td>
        </tr>
        <tr>
          <th>Version</th>
          <td>{{ test_procedure.document.document_control.version }}</td>
        </tr>
        <tr>
          <th>Date</th>
          <td>{{ test_procedure.document.document_control.date }}</td>
        </tr>
        <tr>
          <th>Program</th>
          <td>{{ test_procedure.document.document_control.program }}</td>
        </tr>
        <tr>
          <th>Prepared by</th>
          <td>{{ test_procedure.document.document_control.prepared_by }}</td>
        </tr>
        <tr>
          <th>Approval Authority</th>
          <td>{{ test_procedure.document.document_control.approval_authority }}</td>
        </tr>
      </table>

      <!-- REVISION HISTORY -->
      <h4>Revision History</h4>
      <table class="std-table">
        <thead>
          <tr>
            <th>Version</th>
            <th>Date</th>
            <th>Author</th>
            <th>Changes</th>
          </tr>
        </thead>
        <tbody>
          {% if test_procedure.document.document_control.revision_history and (test_procedure.document.document_control.revision_history|length > 0) %}
            {% for rev in test_procedure.document.document_control.revision_history %}
              <tr>
                <td>{{ rev.version }}</td>
                <td>{{ rev.date }}</td>
                <td>{{ rev.author }}</td>
                <td>{{ rev.changes }}</td>
              </tr>
            {% endfor %}
          {% else %}
            <tr>
              <td>{{ test_procedure.document.document_control.version }}</td>
              <td>{{ test_procedure.document.document_control.date }}</td>
              <td>{{ test_procedure.document.document_control.prepared_by }}</td>
              <td>Initial release</td>
            </tr>
          {% endif %}
        </tbody>
      </table>
    </section>

    

    <!-- 1. INTRODUCTION -->
    <section>
      <h2>1. Introduction</h2>
      <h3>1.1 Purpose</h3>
      <p>{{ test_procedure.introduction.purpose }}</p>

      <h3>1.2 Scope</h3>
      <p>{{ test_procedure.introduction.scope }}</p>
    </section>

    <!-- 2. REFERENCED DOCUMENTS -->
    <section>
      <h2>2. Referenced Documents</h2>
      <p>The following documents are referenced herein and form a part of this test procedure to the extent specified.</p>

      {% if test_procedure.referenced_documents and (test_procedure.referenced_documents|length > 0) %}
        <table class="std-table">
          <thead>
            <tr>
              <th width="25%">Document ID</th>
              <th>Title</th>
              <th width="12%">Version</th>
            </tr>
          </thead>
          <tbody>
            {% for doc in test_procedure.referenced_documents %}
              <tr>
                <td class="proc-id">{{ doc.document_id }}</td>
                <td>{{ doc.title }}</td>
                <td>{{ doc.version if doc.version else "—" }}</td>
              </tr>
            {% endfor %}
          </tbody>
        </table>
      {% elif test_procedure.references and (test_procedure.references|length > 0) %}
        <ul>
          {% for ref in test_procedure.references %}
            <li>{{ ref }}</li>
          {% endfor %}
        </ul>
      {% else %}
        <p class="placeholder">TBD</p>
      {% endif %}
    </section>

    <!-- 3. DEFINITIONS, ACRONYMS, AND ABBREVIATIONS -->
    <section>
      <h2>3. Definitions, Acronyms, and Abbreviations</h2>

      <table class="std-table">
        <thead>
          <tr>
            <th width="25%">Term / Acronym</th>
            <th>Definition</th>
          </tr>
        </thead>
        <tbody>
          <tr><td><strong>IRS</strong></td><td>Interface Requirements Specification — a document defining interface requirements between systems or system elements.</td></tr>
          <tr><td><strong>RVTM</strong></td><td>Requirements Verification Traceability Matrix — a matrix linking requirements to their planned verification activities and results.</td></tr>
          <tr><td><strong>SUT</strong></td><td>System Under Test — the system or system element being evaluated during testing.</td></tr>
          <tr><td><strong>TAID</strong></td><td>Test, Analysis, Inspection, Demonstration — the four standard verification methods used to confirm requirement compliance.</td></tr>
          <tr><td><strong>TBD</strong></td><td>To Be Determined — content that has not yet been defined.</td></tr>
        </tbody>
      </table>
    </section>

    <!-- 4. GENERAL TEST CONDITIONS -->
    <section>
      <h2>4. General Test Conditions</h2>

      <table class="std-table">
        <tr>
          <th width="28%">Test Environment</th>
          <td>{{ test_procedure.general_conditions.test_environment if test_procedure.general_conditions.test_environment else "TBD" }}</td>
        </tr>
        <tr>
          <th>Required Equipment / Tools</th>
          <td>{{ test_procedure.general_conditions.equipment if test_procedure.general_conditions.equipment else "TBD" }}</td>
        </tr>
        <tr>
          <th>Initial System Configuration</th>
          <td>{{ test_procedure.general_conditions.initial_configuration if test_procedure.general_conditions.initial_configuration else "TBD" }}</td>
        </tr>
        <tr>
          <th>Assumptions / Constraints</th>
          <td>{{ test_procedure.general_conditions.assumptions if test_procedure.general_conditions.assumptions else "TBD" }}</td>
        </tr>
      </table>

      <div class="note-box">
        Detailed equipment models, bench configurations, and organization-specific tooling may be tailored by the user where required.
      </div>

      <!-- 4.1 SECURITY CONSIDERATIONS -->
      <h3>4.1 Security Considerations</h3>
      <p>
        {% if test_procedure.general_conditions.security_considerations %}
          {{ test_procedure.general_conditions.security_considerations }}
        {% else %}
          <span class="placeholder">TBD: Security measures, handling requirements for classified or controlled unclassified information (CUI), and cybersecurity considerations applicable during test execution will be defined prior to testing.</span>
        {% endif %}
      </p>

      <!-- 4.2 CONFIGURATION MANAGEMENT -->
      <h3>4.2 Configuration Management</h3>
      <p>
        {% if test_procedure.general_conditions.configuration_management %}
          {{ test_procedure.general_conditions.configuration_management }}
        {% else %}
          <span class="placeholder">TBD: Configuration identification and control procedures for the System Under Test (SUT) will be established prior to test execution. This includes documentation of software build identifiers, hardware serial numbers, firmware versions, and test environment baselines.</span>
        {% endif %}
      </p>
    </section>

    <!-- 5. TEST PROCEDURES -->
    <section>
      <h2>5. Test Procedures</h2>

      {% if test_procedure.procedures and (test_procedure.procedures|length > 0) %}
        {% for proc in test_procedure.procedures %}
          <div class="procedure-block">
            <h3>5.{{ loop.index }} {{ proc.title }}</h3>

            <table class="std-table">
              <tr>
                <th width="25%">Procedure ID</th>
                <td class="proc-id">{{ proc.procedure_id }}</td>
              </tr>
              <tr>
                <th>Linked Requirement ID(s)</th>
                <td class="proc-id">
                  {% if proc.requirement_ids is defined and proc.requirement_ids is iterable and proc.requirement_ids is not string %}
                    {% for rid in proc.requirement_ids %}
                      {{ rid }}{% if not loop.last %}, {% endif %}
                    {% endfor %}
                  {% elif proc.requirement_id is defined %}
                    {{ proc.requirement_id }}
                  {% else %}
                    TBD
                  {% endif %}
                </td>
              </tr>
              <tr>
                <th>Objective</th>
                <td>{{ proc.objective }}</td>
              </tr>
              <tr>
                <th>Verification Method</th>
                <td>{{ proc.verification_method if proc.verification_method else "TBD" }}</td>
              </tr>
              <tr>
                <th>Preconditions</th>
                <td>{{ proc.preconditions if proc.preconditions else "TBD" }}</td>
              </tr>
              <tr>
                <th>Pass / Fail Criteria</th>
                <td>{{ proc.pass_fail_criteria if proc.pass_fail_criteria else "TBD" }}</td>
              </tr>
            </table>

            <h4>Procedure Steps</h4>
            <table class="step-table">
              <thead>
                <tr>
                  <th class="step-number">Step</th>
                  <th>Action</th>
                  <th>Expected Result</th>
                </tr>
              </thead>
              <tbody>
                {% if proc.steps and (proc.steps|length > 0) %}
                  {% for step in proc.steps %}
                    <tr>
                      <td class="step-number">{{ step.step_number }}</td>
                      <td>{{ step.action }}</td>
                      <td>{{ step.expected_result }}</td>
                    </tr>
                  {% endfor %}
                {% else %}
                  <tr>
                    <td class="step-number">1</td>
                    <td>TBD</td>
                    <td>TBD</td>
                  </tr>
                {% endif %}
              </tbody>
            </table>

            <h4>Data Collection / Evidence</h4>
            <p>{{ proc.evidence if proc.evidence else "TBD" }}</p>

            <h4>Notes</h4>
            <p>{{ proc.notes if proc.notes else "" }}</p>
          </div>
        {% endfor %}
      {% else %}
        <p class="placeholder">TBD</p>
      {% endif %}
    </section>

    <!-- 6. TRACEABILITY -->
    <section>
      <h2>6. Traceability</h2>

      <!-- 6.1 VERIFICATION METHOD DEFINITIONS -->
      <h3>6.1 Verification Method Definitions</h3>
      <p>The following verification methods, consistent with standard systems engineering practice (INCOSE SE Handbook; ISO/IEC/IEEE 15288), are referenced throughout this document and the traceability table below.</p>

      <table class="std-table def-table">
        <thead>
          <tr>
            <th width="20%">Method</th>
            <th>Definition</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>Test (T)</td>
            <td>Operation of the system or component under controlled conditions to verify performance against specified requirements. Inputs are provided and outputs are recorded and evaluated against predicted or expected results.</td>
          </tr>
          <tr>
            <td>Analysis (A)</td>
            <td>Use of mathematical modeling, simulation, statistical evaluation, or other technical assessment methods to verify compliance with requirements where physical testing alone is impractical or insufficient.</td>
          </tr>
          <tr>
            <td>Inspection (I)</td>
            <td>Visual examination or review of documentation, design artifacts, or physical hardware to verify conformance with specified requirements without operating the system.</td>
          </tr>
          <tr>
            <td>Demonstration (D)</td>
            <td>Operation of the system to show qualitatively that a required capability exists. Demonstration relies on observable functional performance rather than detailed quantitative measurement.</td>
          </tr>
        </tbody>
      </table>

      <!-- 6.2 PROCEDURE-TO-REQUIREMENT TRACEABILITY -->
      <h3>6.2 Procedure-to-Requirement Traceability</h3>
      <table class="std-table">
        <thead>
          <tr>
            <th>Requirement ID</th>
            <th>Procedure ID</th>
            <th>Verification Method</th>
          </tr>
        </thead>
        <tbody>
          {% if test_procedure.traceability and (test_procedure.traceability|length > 0) %}
            {% for row in test_procedure.traceability %}
              <tr>
                <td class="proc-id">{{ row.requirement_id }}</td>
                <td class="proc-id">{{ row.procedure_id }}</td>
                <td>{{ row.verification_method }}</td>
              </tr>
            {% endfor %}
          {% else %}
            <tr>
              <td class="proc-id">TBD</td>
              <td class="proc-id">TBD</td>
              <td>TBD</td>
            </tr>
          {% endif %}
        </tbody>
      </table>
    </section>

    <!-- 7. DEFECT / NON-CONFORMANCE MANAGEMENT -->
    <section>
      <h2>7. Defect / Non-Conformance Management</h2>

      <p>
        {% if test_procedure.defect_management and test_procedure.defect_management.process %}
          {{ test_procedure.defect_management.process }}
        {% else %}
          <span class="placeholder">TBD: Any test failures, anomalies, or deviations from expected results encountered during procedure execution shall be documented, categorized by severity, and reported to the test lead. The defect record shall include the procedure step at which the failure occurred, observed versus expected behavior, and any relevant data captures or logs.</span>
        {% endif %}
      </p>
    </section>

    <!-- 8. NOTES AND CONSTRAINTS -->
    <section>
      <h2>8. Notes and Constraints</h2>
      <ol>
        {% if test_procedure.notes_and_constraints and (test_procedure.notes_and_constraints|length > 0) %}
          {% for n in test_procedure.notes_and_constraints %}
            <li><strong>{{ n.title }}:</strong> {{ n.description }}</li>
          {% endfor %}
        {% else %}
          <li><strong>TBD:</strong> TBD</li>
        {% endif %}
      </ol>
    </section>

    <!-- SIGNATURES / APPROVALS -->
    <section class="signature-block">
      {% if test_procedure.approvals and test_procedure.approvals.signatories %}
        {% for s in test_procedure.approvals.signatories %}
          <div class="sig-line">
            {{ s.role }}<br />
            __________________<br />
            Date: __________
          </div>
        {% endfor %}
      {% else %}
        <div class="sig-line">
          Test Lead<br />
          __________________<br />
          Date: __________
        </div>
        <div class="sig-line">
          Systems Engineering Lead<br />
          __________________<br />
          Date: __________
        </div>
      {% endif %}
    </section>

    <!-- FOOTER -->
    <div class="footer">
      <p>{{ test_procedure.footer.text }}</p>
    </div>

  </div>
</body>
</html>

"""

tproc_template = Template(TEST_PROCEDURE_TEMPLATE)
tproc_rendered_html = tproc_template.render(test_procedure=test_procedure)

with open("../outputs/TestProcedure_rendered.html", "w", encoding="utf-8") as f:
    f.write(tproc_rendered_html)

print("Test Procedure HTML rendered successfully.")

Test Procedure HTML rendered successfully.


In [46]:
from weasyprint import HTML

HTML("../outputs/TestProcedure_rendered.html").write_pdf("../outputs/TestProcedure_rendered.pdf")

print("Test Procedure PDF generated successfully.")

Test Procedure PDF generated successfully.
